# 06 - Evaluacion final: test, calibracion, explicabilidad y veredicto de hipotesis

Rama: `05FinalModelChecks`. Este cuaderno cierra la Etapa 2 con el modelo ya elegido; **no busca ni entrena modelos nuevos** (salvo las curvas de aprendizaje, que reajustan B4 con fracciones de `train`).

**Protocolo:** `docx/03Evidence/02-11-26_ProtocoloEvaluacionFinal.md`. Fija modelos, intervalos, calibracion y las reglas de H1, H2 y H3 **antes** de abrir `test`. El cuaderno lo aplica; no decide nada nuevo.

**Lo que ya esta hecho (rama 04) y se baja de Kaggle en la celda 1:**

- Escalera B0 a B4 medida en `val` por escenario (A, B y C). B4 (HistGradientBoosting, trial 20) es el modelo principal.
- LightGBM no sustituye a B4; ablaciones por familia y H2 ya ejecutadas en `val`.
- Modelo final para la Etapa 3: B4 ajustado con el `train` de A, en `modelo_final/` (`ptcg_wp.predictor.PredictorVictoria`).
- Evaluacion comun para todos los modelos: `ptcg_wp.evaluation`.

**Orden de este cuaderno (`test` va lo mas tarde posible):**

| Parte | Que hace | Usa | Costo aproximado |
|---|---|---|---|
| 1 | Verifica que los modelos cargados reproducen las metricas de `val` publicadas | `val` | minutos |
| 2 | SHAP y revision de fuga (barrera) | `val` | minutos |
| 3 | Ajuste y decision de calibracion | `calib` | minutos |
| 4 | Curvas de aprendizaje y sobreajuste | `train`, `val` | **horas** |
| 5 | **Evaluacion unica en `test`** y comparacion con referencias | `test` | minutos |
| 6 | Calibracion medida en `test` | `test` | minutos |
| 7 | H1, H2, H3 | `test` | minutos |
| 8 | Analisis de errores | `test` | minutos |
| 9 | Veredicto | resultados | segundos |

Ejecuta las celdas de arriba abajo en un servidor de Colab con CPU + RAM alta. Cada celda que calcula algo revisa primero Kaggle con `store.asegurar`: si sus archivos ya estan, los baja y no recalcula; si faltan, calcula y publica. Asi una sesion cortada se reanuda volviendo a ejecutar desde arriba.

In [ ]:
# PREPARACION COMPLETA DEL CUADERNO (rama 05FinalModelChecks)
# PROPOSITO: dejar este servidor listo con TODO lo producido hasta ahora:
# - Clona o actualiza el repo en la rama 05FinalModelChecks e instala las librerias fijadas
#   (y lightgbm 4.6.0, para que las pruebas de LightGBM tambien corran).
# - Muestra nucleos de CPU y RAM (se espera CPU + RAM alta: 8 nucleos, ~51 GiB).
# - Configura Kaggle y baja del dataset privado del equipo:
#   1) las 21 tablas a /content/datos (estados, partidas y particiones);
#   2) TODOS los resultados del entrenamiento a /content/resultados (B0 a B4, LightGBM,
#      ablaciones, H2, modelo final y READMEs), cada uno verificado con su SHA-256.
#   Lo que ya este en este servidor no se vuelve a bajar.
# ESPERADO: la linea de nucleos/RAM, "Tablas listas | caracteristicas: 62" y un resumen de
# resultados por carpeta, con "Todos los resultados del indice estan en /content/resultados".
import json
import os
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "05FinalModelChecks"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt lightgbm==4.6.0
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m == "ptcg_wp" or m.startswith("ptcg_wp.")]:
    del sys.modules[nombre]

# Nucleos y RAM de este servidor, y credenciales de Kaggle
from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.7.2", (f"Se requiere scikit-learn 1.7.2; version cargada: {sklearn_version}. "
                                    "Reinicia el runtime y ejecuta esta celda de nuevo.")
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days

# 1) Tablas procesadas -> /content/datos
DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62, len(FEATURES)

# 2) TODOS los resultados publicados (segun el indice del dataset) -> /content/resultados
indice = store.leer_indice(DATASET)
TODOS_LOS_RESULTADOS = sorted(r for r, fila in indice.items() if fila["tipo"] == "resultado")
assert store.listo(TODOS_LOS_RESULTADOS), "Hay resultados del indice que no se pudieron bajar"
por_carpeta = Counter("/".join(r.split("/")[:2]) if r.count("/") >= 2 else r.split("/")[0]
                      for r in TODOS_LOS_RESULTADOS)
display(pd.DataFrame(sorted(por_carpeta.items()), columns=["carpeta", "archivos"]))
print(f"Todos los resultados del indice estan en {RESULTADOS}: {len(TODOS_LOS_RESULTADOS)} archivos")

# Constantes y trazabilidad para el resto del cuaderno
SEED = 2026
SCENARIOS = ("A", "B", "C")
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
RUN_INFO = {"commit": COMMIT, "working_tree_dirty": DIRTY, "seed": SEED,
            "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "recursos": RECURSOS}
print("Tablas listas | caracteristicas:", len(FEATURES), "| rama:", BRANCH, "| commit:", COMMIT[:7],
      "| cambios locales:", DIRTY)

In [ ]:
# PRUEBAS AUTOMATICAS DEL REPO
# PROPOSITO: confirmar que el codigo de esta rama esta sano ANTES de tocar test. Usan datos
# sinteticos y no descargan nada (unos 2 minutos). Como la celda anterior bajo todos los
# resultados e instalo lightgbm, aqui no se omite ninguna prueba.
# - Fuga y etiquetas, particiones, carga de datos y almacen de Kaggle.
# - Entrenamiento por lotes y protocolo agrupado de B2, B3 y B4.
# - Evaluacion comun igual a scikit-learn; H2 sin perdida de categorias; LightGBM sin avisos.
# - Modelo final: la observacion produce las mismas caracteristicas que el entrenamiento.
# - Evidencia de B2/B3 contra los resultados bajados en /content/resultados.
# - Evaluacion final: calib y test cerrados por defecto, intervalos, calibracion y reglas H1-H3.
# ESPERADO: "109 passed".
# Si alguna dice "failed", NO seguir: avisar al equipo con el nombre de la prueba.
!python -m pytest -q tests

## Parte 0: barreras antes de empezar

La evaluacion en `test` solo vale si el protocolo se escribio antes y si el codigo que corre es el versionado. Esta celda lo comprueba y deja listas las piezas comunes.

In [ ]:
# BARRERAS Y PIEZAS COMUNES
# PROPOSITO: comprobar que el protocolo esta en el repo y que este servidor corre el commit
# subido (sin cambios locales). Define rutas y funciones que usa el resto del cuaderno.
# No calcula nada. ESPERADO: "Protocolo ... congelado en el commit ...".
import gc
import joblib
import matplotlib.pyplot as plt
from ptcg_wp import evaluacion_final as ef
from ptcg_wp.evaluation import STAGES
from ptcg_wp.predictor import MODELO_EXPORTADO

PROTOCOLO = REPO_DIR / "docx/03Evidence/02-11-26_ProtocoloEvaluacionFinal.md"
assert PROTOCOLO.exists(), "Falta el protocolo: no se abre test sin el"
assert not DIRTY, "Hay cambios locales en /content/repo: los resultados no serian trazables"
BASE = "evaluacion_final"
MODELOS = list(ef.MODELOS)
CALIBRADOS = ("B2", "B3", "B4")
ESCENARIOS_CALIB = ("A", "B")
CACHE = store.TRABAJO_DIR
N_PARTIDAS_SHAP = 1000
NUCLEOS = RECURSOS["nucleos_usables"]


def ruta(*partes):
    return "/".join((BASE, *partes))


def local(relativa):
    salida = RESULTADOS / relativa
    salida.parent.mkdir(parents=True, exist_ok=True)
    return salida


def predicciones(particion, escenario):
    return pd.read_parquet(RESULTADOS / ruta("predicciones", f"{particion}_{escenario}.parquet"))


print(f"Protocolo {PROTOCOLO.name} congelado en el commit {COMMIT[:7]}")

## Parte 1: verificacion de los modelos en `val`

Antes de gastar `test` se comprueba que este cuaderno carga bien cada modelo. Las predicciones de B0 a B4 y H2 sobre `val` deben reproducir las metricas que los cuadernos 02 a 05 publicaron. Si no coinciden, hay un error de carga y `test` no se toca.

In [ ]:
# VERIFICAR LA CARGA DE MODELOS CONTRA LAS METRICAS DE VAL PUBLICADAS
# PROPOSITO: predecir val con B0-B4 y H2 y compararlo con los CSV de las ramas anteriores.
# Publica evaluacion_final/predicciones/val_X.parquet. NO toca calib ni test.
# ESPERADO: por escenario "OK": log loss y Brier iguales a 1e-9 y AUC a 1e-6. B3 se tolera hasta
# 1e-3 solo en AUC: tiene empates exactos y su AUC guardado depende del tamano de lote (ver
# verificar_contra_val); que sus predicciones son las mismas lo prueban log loss y Brier.
for escenario in SCENARIOS:
    archivo = ruta("predicciones", f"val_{escenario}.parquet")
    print(f"--- Escenario {escenario} ---")

    def predecir_val(escenario=escenario, archivo=archivo):
        tabla = ef.predicciones_particion(DATA_DIR, RESULTADOS, escenario, "val", FEATURES)
        tabla.to_parquet(local(archivo), index=False)

    store.asegurar([archivo], predecir_val, f"Predicciones de val escenario {escenario}")
    verificacion = ef.verificar_contra_val(predicciones("val", escenario), RESULTADOS, escenario)
    display(verificacion.loc[verificacion["etapa"].eq("global")].round(6))
    es_b3 = verificacion["modelo"].eq("B3")
    dif_auc, dif_b3 = verificacion.loc[~es_b3, "dif_auc"].max(), verificacion.loc[es_b3, "dif_auc"].max()
    dif_puntajes = verificacion["dif_puntajes"].max()
    assert dif_puntajes < 1e-9 and dif_auc < 1e-6 and dif_b3 < 1e-3, (
        f"{escenario}: no reproduce val ({dif_puntajes}, {dif_auc}, {dif_b3})")
    print(f"OK {escenario}: log loss y Brier {dif_puntajes:.2e} | AUC {dif_auc:.2e} | AUC de B3 {dif_b3:.2e}")

## Parte 2: SHAP y revision de fuga (barrera)

Se explica el modelo final (B4 de A) sobre una muestra de **partidas completas** de `val`. Va antes de `test` a proposito: si apareciera una fuga, `test` seguiria intacto. La revision aprueba si ninguna columna prohibida esta en el modelo, ninguna caracteristica concentra mas del 50 % de la importancia y ninguna predice sola con AUC de 0.85 o mas.

In [ ]:
# SHAP DEL MODELO FINAL Y REVISION DE FUGA
# PROPOSITO: valores SHAP de una muestra de partidas de val de A, importancia global y por etapa,
# AUC de cada caracteristica sola y la decision de la revision de fuga. Publica evaluacion_final/shap/.
# Unos minutos con ocho nucleos. test NO participa.
# ESPERADO: la tabla de importancia y "Revision de fuga aprobada: True".
ARCHIVOS_SHAP = [ruta("shap", n) for n in ("importancia.csv", "auc_individual.csv", "revision_fuga.json",
                                           "valores.parquet", "muestra.parquet")]


def calcular_shap():
    artefacto = joblib.load(RESULTADOS / MODELO_EXPORTADO)
    assert list(artefacto["features"]) == FEATURES
    tabla = ef.cargar_particion(DATA_DIR, "A", "val", FEATURES)
    x = tabla[FEATURES].to_numpy(dtype=np.float32)
    individual = ef.auc_individual(x, tabla["y"].to_numpy(), ef.codigos_partida(tabla["game_id"]), FEATURES)
    elegidas = tabla["game_id"].isin(ef.muestra_partidas(tabla["game_id"], N_PARTIDAS_SHAP, SEED)).to_numpy()
    muestra = tabla.loc[elegidas].reset_index(drop=True)
    valores, base = ef.valores_shap(artefacto["model"], x[elegidas], n_jobs=NUCLEOS)
    importancia = ef.importancia_shap(valores, FEATURES, muestra["stage"].to_numpy(),
                                      muestra["game_id"].to_numpy())
    revision = ef.revision_fuga(importancia, individual, FEATURES)
    importancia.to_csv(local(ARCHIVOS_SHAP[0]), index=False)
    individual.to_csv(local(ARCHIVOS_SHAP[1]), index=False)
    local(ARCHIVOS_SHAP[2]).write_text(json.dumps(
        {**revision, "valor_base_logit": base, "partidas": int(muestra["game_id"].nunique()),
         "estados": len(muestra), **RUN_INFO}, indent=2, ensure_ascii=False))
    pd.DataFrame(valores.astype(np.float32), columns=FEATURES).to_parquet(local(ARCHIVOS_SHAP[3]), index=False)
    muestra[["game_id", "stage", "y", *FEATURES]].to_parquet(local(ARCHIVOS_SHAP[4]), index=False)


store.asegurar(ARCHIVOS_SHAP, calcular_shap, "SHAP del modelo final y revision de fuga")
importancia = pd.read_csv(RESULTADOS / ARCHIVOS_SHAP[0])
individual = pd.read_csv(RESULTADOS / ARCHIVOS_SHAP[1])
revision = json.loads((RESULTADOS / ARCHIVOS_SHAP[2]).read_text())
globales = importancia.loc[importancia["etapa"].eq("global")]
display(globales.head(15).round(4))
display(globales.groupby("familia")["fraccion"].sum().sort_values(ascending=False).round(3).to_frame())
display(importancia.pivot(index="caracteristica", columns="etapa", values="fraccion")
        .loc[globales["caracteristica"].head(12), ["inicio", "medio", "final", "global"]].round(3))
display(individual.head(5).round(4))
print("Revision:", {k: revision[k] for k in ("sin_columnas_prohibidas", "ninguna_domina", "ninguna_predice_sola",
                                             "mas_importante", "fraccion_mas_importante", "auc_mejor_individual")})
print("Revision de fuga aprobada:", revision["aprobada"])

In [ ]:
# FIGURA SHAP (SOLO LECTURA)
# PROPOSITO: dibujar como empuja cada caracteristica la prediccion. No calcula nada nuevo.
import shap

valores_df = pd.read_parquet(RESULTADOS / ARCHIVOS_SHAP[3])
muestra_df = pd.read_parquet(RESULTADOS / ARCHIVOS_SHAP[4])
shap.summary_plot(valores_df.to_numpy(), muestra_df[FEATURES], feature_names=FEATURES, max_display=15, show=False)
plt.title("SHAP de B4 (A) en una muestra de partidas de val")
plt.tight_layout()
plt.show()

## Parte 3: calibracion con `calib`

Se mide antes de corregir, como pide el articulo. Dentro de `calib` y por pliegues de partida se comparan sin correccion, isotonica y Platt para B2, B3 y B4 en A y B. La eleccion queda escrita aqui, **antes de abrir `test`**. Los calibradores solo ven `calib`.

In [ ]:
# AJUSTE Y DECISION DE CALIBRACION (SOLO CALIB)
# PROPOSITO: por escenario A y B, predecir calib, comparar metodos por validacion cruzada agrupada
# y ajustar los calibradores con todo calib. Publica evaluacion_final/predicciones/calib_X.parquet y
# evaluacion_final/calibracion/. test NO participa.
# ESPERADO: una tabla por escenario y la decision de cada modelo.
for escenario in ESCENARIOS_CALIB:
    archivos = [ruta("predicciones", f"calib_{escenario}.parquet"),
                ruta("calibracion", f"validacion_{escenario}.csv"),
                ruta("calibracion", f"calibradores_{escenario}.joblib"),
                ruta("calibracion", f"decision_{escenario}.json")]
    print(f"--- Escenario {escenario} ---")

    def calibrar(escenario=escenario, archivos=archivos):
        tabla = ef.predicciones_particion(DATA_DIR, RESULTADOS, escenario, "calib", FEATURES)
        tabla.to_parquet(local(archivos[0]), index=False)
        y = tabla["y"].to_numpy()
        codigos = ef.codigos_partida(tabla["game_id"])
        pesos = ef.pesos_partida(codigos)
        filas, calibradores, decision = [], {}, {}
        for modelo in CALIBRADOS:
            p = tabla[f"p_{modelo}"].to_numpy()
            validacion = ef.validar_calibradores(p, y, codigos, seed=SEED)
            filas.append(validacion.assign(escenario=escenario, modelo=modelo))
            decision[modelo] = ef.elegir_metodo(validacion)
            calibradores[modelo] = {m: ef.ajustar_calibrador(m, p, y, pesos) for m in ef.METODOS}
        pd.concat(filas, ignore_index=True).to_csv(local(archivos[1]), index=False)
        joblib.dump(calibradores, local(archivos[2]))
        local(archivos[3]).write_text(json.dumps({"escenario": escenario, "decision": decision, **RUN_INFO},
                                                 indent=2, ensure_ascii=False))

    store.asegurar(archivos, calibrar, f"Calibracion en calib escenario {escenario}")
    validacion = pd.read_csv(RESULTADOS / archivos[1])
    display(validacion[["modelo", "metodo", "auc", "log_loss", "brier", "n_partidas"]].round(5))
    print("Decision:", json.loads((RESULTADOS / archivos[3]).read_text())["decision"])

## Parte 4: curvas de aprendizaje

Reajusta B4 con el 25, 50 y 75 % de las partidas de `train` (anidadas) con el mismo protocolo del modelo final; el 100 % reutiliza ese modelo. Se mide en `val` y en una muestra fija de `train`. **Es lo mas caro del cuaderno: unas 3 h 45 min en total**, y cada escenario se publica al terminar para poder reanudar.

In [ ]:
# CURVAS DE APRENDIZAJE Y DIAGNOSTICO DE SOBREAJUSTE
# PROPOSITO: por escenario, bajar la curva de Kaggle si existe; si falta, reajustar B4 con 25/50/75 %
# de train (unos 75 min por escenario) y publicarla. test NO participa.
# ESPERADO: el 100 % reproduce el AUC de val de B4 (A 0.8346, B 0.7863, C 0.8355).
assert store.listo(["b4/busqueda/b4_candidatas.csv"])
candidata_b4 = pd.read_csv(RESULTADOS / "b4/busqueda/b4_candidatas.csv")
candidata_b4 = candidata_b4.loc[candidata_b4["trial"].eq(20)].copy()
assert len(candidata_b4) == 1
curvas = []
for escenario in SCENARIOS:
    archivo = ruta("curvas", f"curva_{escenario}.csv")
    print(f"--- Curva escenario {escenario} ---")

    def calcular_curva(escenario=escenario, archivo=archivo):
        curva = ef.curva_aprendizaje(DATA_DIR, escenario, FEATURES, candidata_b4,
                                     RESULTADOS / ef.MODELO_B4.format(e=escenario), seed=SEED, cache_dir=CACHE)
        curva.to_csv(local(archivo), index=False)

    store.asegurar([archivo], calcular_curva, f"Curva de aprendizaje escenario {escenario}")
    curva = pd.read_csv(RESULTADOS / archivo)
    guardado = ef.metricas_val_guardadas(RESULTADOS, escenario).query("modelo == 'B4' and etapa == 'global'")
    completo = curva.query("fraccion == 1.0 and conjunto == 'val' and etapa == 'global'")
    assert abs(completo["auc"].iloc[0] - guardado["auc"].iloc[0]) < 1e-6, "El 100 % no reproduce B4"
    curvas.append(curva)
curvas = pd.concat(curvas, ignore_index=True)
diagnostico = ef.diagnostico_sobreajuste(curvas)
display(diagnostico.round(4))
fig, ejes = plt.subplots(1, len(SCENARIOS), figsize=(12, 3.6), sharey=True)
for eje, escenario in zip(ejes, SCENARIOS):
    parte = diagnostico.loc[diagnostico["escenario"].eq(escenario)]
    eje.plot(parte["partidas_train"], parte["auc_train"], marker="o", label="train")
    eje.plot(parte["partidas_train"], parte["auc_val"], marker="o", label="val")
    eje.set_title(f"Escenario {escenario}")
    eje.set_xlabel("Partidas de train")
ejes[0].set_ylabel("AUC")
ejes[0].legend()
plt.tight_layout()
plt.show()

## Parte 5: evaluacion unica en `test`

**Desde aqui se abre `test`.** La celda no corre si falta alguna barrera. Una vez publicadas, las predicciones quedan en Kaggle y no se vuelven a calcular. Ningun numero de aqui en adelante puede cambiar un modelo, un calibrador ni una regla.

In [ ]:
# EVALUACION UNICA EN TEST
# PROPOSITO: predecir test una sola vez con los modelos de cada escenario y medir AUC, log loss y
# Brier, globales y por etapa, con intervalos por remuestreo de partidas (1000, percentil 95 %).
# Barreras: revision de fuga aprobada y decision de calibracion publicada.
# Publica evaluacion_final/test/ y evaluacion_final/predicciones/test_X.parquet.
# ESPERADO: una tabla por escenario. A, B y C NO se promedian.
assert revision["aprobada"], "La revision de fuga no aprobo: NO abrir test; avisar al equipo"
assert store.listo([ruta("calibracion", f"decision_{e}.json") for e in ESCENARIOS_CALIB])
assert store.listo([ruta("curvas", f"curva_{e}.csv") for e in SCENARIOS])
metricas_test, comparaciones_test = {}, {}
for escenario in SCENARIOS:
    archivos = [ruta("predicciones", f"test_{escenario}.parquet"), ruta("test", f"metricas_{escenario}.csv"),
                ruta("test", f"comparaciones_{escenario}.csv"), ruta("test", f"registro_{escenario}.json")]
    print(f"--- Escenario {escenario} ---")

    def evaluar_test(escenario=escenario, archivos=archivos):
        tabla = ef.predicciones_particion(DATA_DIR, RESULTADOS, escenario, "test", FEATURES)
        tabla.to_parquet(local(archivos[0]), index=False)
        comun = {"escenario": escenario, "particion": "test"}
        metricas, replicas = ef.evaluar_con_intervalos(tabla, MODELOS, comun, n_boot=ef.N_BOOT, seed=SEED)
        comparacion = ef.comparar(metricas, replicas, "B4", ["B0", "B1", "B2", "B3", "H2"], comun)
        metricas.to_csv(local(archivos[1]), index=False)
        comparacion.to_csv(local(archivos[2]), index=False)
        local(archivos[3]).write_text(json.dumps(
            {**RUN_INFO, "escenario": escenario, "protocolo": PROTOCOLO.name, "protocolo_sha256": store.sha256(PROTOCOLO),
             "n_boot": ef.N_BOOT, "partidas": int(tabla["game_id"].nunique()), "estados": len(tabla),
             "fraccion_ids_desconocidos": float(tabla["id_desconocido"].mean())}, indent=2, ensure_ascii=False))

    store.asegurar(archivos, evaluar_test, f"Evaluacion unica en test escenario {escenario}")
    metricas_test[escenario] = pd.read_csv(RESULTADOS / archivos[1])
    comparaciones_test[escenario] = pd.read_csv(RESULTADOS / archivos[2])
    columnas = ["modelo", "etapa", "auc", "auc_ic_inf", "auc_ic_sup", "log_loss", "log_loss_ic_inf", "log_loss_ic_sup",
                "brier", "brier_ic_inf", "brier_ic_sup", "n_estados", "n_partidas"]
    display(metricas_test[escenario][columnas].round(4))

### Comparacion contra las referencias

B0 es el piso trivial y B2 la referencia razonable. B1 es el criterio interno de exito de la Seccion III-D del articulo. La banda de literatura (AUC 0.78-0.80) es un orden de magnitud, no un umbral: difieren el juego, la poblacion y la particion.

In [ ]:
# COMPARACION CONTRA B0, B1, B2 Y LITERATURA
# PROPOSITO: diferencias emparejadas de B4 frente a las referencias y curva de AUC por turno en A.
# Publica evaluacion_final/test/auc_por_turno_A.csv y referencias.json.
# ESPERADO: una tabla de diferencias por escenario y el turno en que A alcanza la banda, si la alcanza.
BANDA_LITERATURA = (0.78, 0.80)
ARCHIVOS_REFERENCIAS = [ruta("test", "auc_por_turno_A.csv"), ruta("test", "referencias.json")]


def calcular_referencias():
    curva = ef.auc_por_turno(predicciones("test", "A"), "B4", n_boot=ef.N_BOOT, seed=SEED)
    curva.to_csv(local(ARCHIVOS_REFERENCIAS[0]), index=False)
    final_a = metricas_test["A"].query("modelo == 'B4' and etapa == 'final'").iloc[0]
    resumen = {"criterio_b1": {e: ef.criterio_b1(metricas_test[e]) for e in SCENARIOS},
               "literatura": {"banda": BANDA_LITERATURA, "auc_final_A": float(final_a["auc"]),
                              "ic": [float(final_a["auc_ic_inf"]), float(final_a["auc_ic_sup"])],
                              "alcanza_banda": bool(final_a["auc"] >= BANDA_LITERATURA[0]),
                              "primer_turno": ef.primer_turno_banda(curva, BANDA_LITERATURA[0]),
                              "nota": "orden de magnitud; no es comparable de forma estricta"}}
    local(ARCHIVOS_REFERENCIAS[1]).write_text(json.dumps(resumen, indent=2, ensure_ascii=False))


store.asegurar(ARCHIVOS_REFERENCIAS, calcular_referencias, "Referencias y AUC por turno en test")
referencias = json.loads((RESULTADOS / ARCHIVOS_REFERENCIAS[1]).read_text())
for escenario in SCENARIOS:
    print(f"--- Escenario {escenario}: B4 menos referencia ---")
    tabla = comparaciones_test[escenario].query("referencia in ['B0', 'B1', 'B2', 'B3'] and etapa == 'global'")
    display(tabla[["referencia", "metrica", "valor_principal", "valor_referencia", "delta", "ic_inf", "ic_sup"]].round(4))
print(json.dumps(referencias, indent=2, ensure_ascii=False))
curva_turno = pd.read_csv(RESULTADOS / ARCHIVOS_REFERENCIAS[0])
plt.figure(figsize=(7, 3.6))
plt.plot(curva_turno["turno"], curva_turno["auc"], marker="o", label="B4 en test de A")
plt.fill_between(curva_turno["turno"], curva_turno["auc_ic_inf"], curva_turno["auc_ic_sup"], alpha=0.2)
plt.axhspan(*BANDA_LITERATURA, color="gray", alpha=0.2, label="Literatura (orden de magnitud)")
plt.xlabel("Turno (el ultimo punto agrupa 20 o mas)")
plt.ylabel("AUC")
plt.legend()
plt.tight_layout()
plt.show()

## Parte 6: calibracion medida en `test`

Fiabilidad, Brier, log loss, ECE y descomposicion CORP antes y despues, en A y en B. Se muestran todos los metodos ajustados en `calib`; verlos aqui no cambia la decision ya publicada en la Parte 3.

In [ ]:
# CALIBRACION EN TEST: ANTES Y DESPUES
# PROPOSITO: aplicar a test los calibradores ajustados con calib y medir el efecto por modelo.
# Publica evaluacion_final/calibracion/test_X.csv, corp_X.csv y fiabilidad_X.csv.
# ESPERADO: una tabla por escenario con Brier antes y despues, y los diagramas de fiabilidad.
tablas_calibracion, tablas_corp, tablas_fiabilidad = {}, {}, {}
for escenario in ESCENARIOS_CALIB:
    archivos = [ruta("calibracion", f"{n}_{escenario}.csv") for n in ("test", "corp", "fiabilidad")]
    print(f"--- Escenario {escenario} ---")

    def medir_calibracion(escenario=escenario, archivos=archivos):
        tabla = predicciones("test", escenario)
        calibradores = joblib.load(RESULTADOS / ruta("calibracion", f"calibradores_{escenario}.joblib"))
        y = tabla["y"].to_numpy()
        codigos = ef.codigos_partida(tabla["game_id"])
        variantes, corp, fiabilidad = [], [], []
        for modelo in CALIBRADOS:
            for metodo in ef.METODOS:
                nombre = f"{modelo}_{metodo}"
                tabla[f"p_{nombre}"] = ef.aplicar_calibrador(calibradores[modelo][metodo], tabla[f"p_{modelo}"])
                variantes.append(nombre)
                comun = {"escenario": escenario, "modelo": modelo, "metodo": metodo}
                corp.append({**comun, **ef.descomposicion_corp(y, tabla[f"p_{nombre}"].to_numpy(), codigos)})
                bins = ef.tabla_fiabilidad(y, tabla[f"p_{nombre}"].to_numpy(), codigos)
                corp[-1]["ece"] = ef.ece(bins)
                fiabilidad.append(bins.assign(**comun))
        metricas, _ = ef.evaluar_con_intervalos(tabla, variantes, {"escenario": escenario}, n_boot=ef.N_BOOT, seed=SEED)
        metricas.to_csv(local(archivos[0]), index=False)
        pd.DataFrame(corp).to_csv(local(archivos[1]), index=False)
        pd.concat(fiabilidad, ignore_index=True).to_csv(local(archivos[2]), index=False)

    store.asegurar(archivos, medir_calibracion, f"Calibracion medida en test escenario {escenario}")
    tablas_calibracion[escenario] = pd.read_csv(RESULTADOS / archivos[0])
    tablas_corp[escenario] = pd.read_csv(RESULTADOS / archivos[1])
    tablas_fiabilidad[escenario] = pd.read_csv(RESULTADOS / archivos[2])
    display(tablas_corp[escenario].round(5))
    display(tablas_calibracion[escenario].query("etapa == 'global'")[
        ["modelo", "auc", "log_loss", "log_loss_ic_inf", "log_loss_ic_sup", "brier", "brier_ic_inf", "brier_ic_sup"]].round(5))

fig, ejes = plt.subplots(1, len(ESCENARIOS_CALIB), figsize=(9, 4.2), sharey=True)
for eje, escenario in zip(ejes, ESCENARIOS_CALIB):
    parte = tablas_fiabilidad[escenario].query("modelo == 'B4'")
    for metodo, grupo in parte.groupby("metodo"):
        eje.plot(grupo["p_media"], grupo["y_media"], marker="o", label=metodo)
    eje.plot([0, 1], [0, 1], color="gray", linestyle="--")
    eje.set_title(f"B4 en test de {escenario}")
    eje.set_xlabel("Probabilidad predicha")
ejes[0].set_ylabel("Frecuencia observada")
ejes[0].legend()
plt.tight_layout()
plt.show()

## Parte 7: hipotesis

Las reglas estan en la seccion 7 del protocolo. **H3 no compara directamente la caida de AUC con la de log loss**, porque estan en escalas distintas: se decide con la descomposicion `puntaje = MCB - DSC + UNC`, que deja calibracion perdida y ordenamiento perdido en las mismas unidades, sobre las mismas partidas. Las caidas directas se muestran tambien, como pide el plan, pero no deciden.

In [ ]:
# H1 Y H2
# PROPOSITO: aplicar las reglas del protocolo por escenario con B4 sin recalibrar.
# Publica evaluacion_final/hipotesis/h1.csv y h2.csv.
ARCHIVOS_H12 = [ruta("hipotesis", "h1.csv"), ruta("hipotesis", "h2.csv")]


def probar_h1_h2():
    pd.DataFrame([ef.prueba_h1(predicciones("test", e), e, n_boot=ef.N_BOOT, seed=SEED)
                  for e in SCENARIOS]).to_csv(local(ARCHIVOS_H12[0]), index=False)
    pd.DataFrame([ef.decidir_h2(comparaciones_test[e], e) for e in SCENARIOS]).to_csv(local(ARCHIVOS_H12[1]), index=False)


store.asegurar(ARCHIVOS_H12, probar_h1_h2, "H1 y H2 en test")
h1 = pd.read_csv(RESULTADOS / ARCHIVOS_H12[0])
h2 = pd.read_csv(RESULTADOS / ARCHIVOS_H12[1])
display(h1.round(4))
display(h2.round(4))

In [ ]:
# H3: DEGRADACION DE CALIBRACION Y ORDENAMIENTO BAJO CAMBIO TEMPORAL
# PROPOSITO: mismas partidas (test de A jugadas el 15 y 22 jul, que tambien son test de B), modelo de
# A contra modelo de B. Agrega la serie por dia del modelo de B y la comparacion no emparejada.
# Publica evaluacion_final/hipotesis/h3.csv, h3_decision.json y h3_serie_diaria.csv.
ARCHIVOS_H3 = [ruta("hipotesis", n) for n in ("h3.csv", "h3_decision.json", "h3_serie_diaria.csv")]


def probar_h3():
    filas, decision = ef.prueba_h3(predicciones("test", "A"), predicciones("test", "B"), n_boot=ef.N_BOOT, seed=SEED)
    filas.to_csv(local(ARCHIVOS_H3[0]), index=False)
    local(ARCHIVOS_H3[1]).write_text(json.dumps(decision, indent=2, ensure_ascii=False))
    ef.serie_diaria([predicciones(p, "B") for p in ("val", "calib", "test")]).to_csv(local(ARCHIVOS_H3[2]), index=False)


store.asegurar(ARCHIVOS_H3, probar_h3, "H3 en test")
h3 = pd.read_csv(RESULTADOS / ARCHIVOS_H3[0])
h3_decision = json.loads((RESULTADOS / ARCHIVOS_H3[1]).read_text())
h3_serie = pd.read_csv(RESULTADOS / ARCHIVOS_H3[2])
display(h3.round(5))
print("H3:", h3_decision["resultado"], "| partidas comunes:", h3_decision["partidas_comunes"])
display(h3_serie[["dia", "auc", "log_loss", "brier", "log_loss_mcb", "log_loss_dsc", "brier_mcb", "brier_dsc", "n_partidas"]].round(5))
global_ab = pd.concat([metricas_test[e].query("modelo == 'B4' and etapa == 'global'") for e in ("A", "B")])
print("Referencia NO emparejada (poblaciones distintas):")
display(global_ab[["escenario", "auc", "log_loss", "brier", "n_partidas"]].round(4))
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.4))
ejes[0].plot(h3_serie["dia"], h3_serie["auc"], marker="o")
ejes[0].set_title("AUC del modelo de B por dia")
ejes[1].plot(h3_serie["dia"], h3_serie["log_loss_mcb"], marker="o", label="MCB (calibracion)")
ejes[1].plot(h3_serie["dia"], h3_serie["log_loss_dsc"], marker="o", label="DSC (ordenamiento)")
ejes[1].set_title("Descomposicion del log loss")
ejes[1].legend()
plt.tight_layout()
plt.show()

## Parte 8: analisis de errores

Es exploratorio y va despues de `test`: describe, no cambia ningun modelo. Falsos positivos y negativos con umbral 0.5, y predicciones muy seguras (p >= 0.9 o p <= 0.1) y equivocadas, por etapa.

In [ ]:
# ANALISIS DE ERRORES
# PROPOSITO: tasas de error por etapa en cada escenario y perfil de los errores muy seguros.
# Publica evaluacion_final/errores/errores_X.csv y perfil_X.csv.
for escenario in SCENARIOS:
    archivos = [ruta("errores", f"errores_{escenario}.csv"), ruta("errores", f"perfil_{escenario}.csv")]
    print(f"--- Escenario {escenario} ---")

    def analizar_errores(escenario=escenario, archivos=archivos):
        tabla = predicciones("test", escenario)
        ef.tabla_errores(tabla, escenario).to_csv(local(archivos[0]), index=False)
        columnas = ef.cargar_particion(DATA_DIR, escenario, "test", ["step", *FEATURES])
        unidas = tabla.merge(columnas[["game_id", "step", *FEATURES]], on=["game_id", "step"], validate="one_to_one")
        assert len(unidas) == len(tabla)
        ef.perfil_errores(unidas, FEATURES).to_csv(local(archivos[1]), index=False)

    store.asegurar(archivos, analizar_errores, f"Analisis de errores escenario {escenario}")
    errores = pd.read_csv(RESULTADOS / archivos[0])
    display(errores[["etapa", "tasa_falso_positivo", "tasa_falso_negativo", "tasa_seguro_equivocado_alto",
                     "tasa_seguro_equivocado_bajo", "precision_seguras", "n_partidas"]].round(4))
    display(pd.read_csv(RESULTADOS / archivos[1]).round(3))

## Parte 9: veredicto

Reune lo que dictan las reglas. Los intervalos son individuales por escenario, no simultaneos, y no incorporan la incertidumbre de la busqueda ni del ajuste. El alcance es el de los datos: agentes automaticos de una franja de puntuacion, diez dias y las versiones del motor presentes, con calendario y motor confundidos.

In [ ]:
# VEREDICTO SOBRE H1, H2 Y H3
# PROPOSITO: juntar los resultados de las reglas en un solo archivo. Publica evaluacion_final/veredicto.json.
ARCHIVO_VEREDICTO = ruta("veredicto.json")


def escribir_veredicto():
    veredicto = {"H1": h1.set_index("escenario")["resultado"].to_dict(),
                 "H2": h2.set_index("escenario")["resultado"].to_dict(),
                 "H3": h3_decision["resultado"], "criterio_b1": referencias["criterio_b1"],
                 "literatura": referencias["literatura"], "revision_fuga": revision["aprobada"],
                 "protocolo": PROTOCOLO.name, **RUN_INFO}
    local(ARCHIVO_VEREDICTO).write_text(json.dumps(veredicto, indent=2, ensure_ascii=False))


store.asegurar([ARCHIVO_VEREDICTO], escribir_veredicto, "Veredicto de hipotesis")
veredicto = json.loads((RESULTADOS / ARCHIVO_VEREDICTO).read_text())
print(json.dumps({k: veredicto[k] for k in ("H1", "H2", "H3", "criterio_b1", "literatura")}, indent=2, ensure_ascii=False))

## Interpretacion del equipo (escribir despues de ejecutar)

Esta celda no se puede llenar antes de ver los resultados y no debe llenarse con cifras que no salgan de las celdas de arriba.

1. **SHAP, hallazgos accionables:** que caracteristicas dominan, como cambian por etapa y que implica para la Etapa 3.
2. **Sobreajuste:** brecha `train - val` y si mas partidas ayudarian.
3. **Calibracion:** si la correccion elegida en `calib` mejoro `test` en A y en B, y si Platt o la isotonica transfieren mejor a B.
4. **Errores:** en que situaciones falla el modelo con mas seguridad.
5. **Veredicto:** H1, H2 y H3 con su alcance y limitaciones.